# LSTM — Long Short-Term Memory for Spike Classification
Models temporal dependencies through recurrent state rather than manual lag features.
Manual spike_lag and price_lag columns are dropped; the sequence window captures this context.
The lookback window length is included in the hyperparameter search.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc

from shared.data_prep import (
    load_data, split_data, get_feature_cols, fit_scaler, apply_scaler,
    compute_pos_weight, random_search, SequenceDataset,
    train_model, evaluate, TARGET
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

## 1. Data Loading
LSTM uses a reduced feature set — manual lag features (spike_lag_*, price_lag_*) are omitted because the recurrent state captures temporal dependencies from the sequence itself.

In [ ]:
df = load_data()
train, val, test = split_data(df)

# LSTM feature set: excludes spike_lag_* and price_lag_* (redundant with the sequence)
feature_cols = get_feature_cols("LSTM", df.columns.tolist())
N_FEATURES = len(feature_cols)
print(f"LSTM features: {N_FEATURES}  (manual lags dropped)")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

scaler, cont_cols = fit_scaler(train, feature_cols)
train_s = apply_scaler(train, scaler, cont_cols)
val_s   = apply_scaler(val,   scaler, cont_cols)
test_s  = apply_scaler(test,  scaler, cont_cols)

pos_weight = compute_pos_weight(train[TARGET])
print(f"pos_weight: {pos_weight.item():.2f}")

## 2. Model Architecture

In [ ]:
class LSTMClassifier(nn.Module):
    """
    Stacked LSTM for binary classification.
    Input : (batch, lookback, n_features)
    Output: single logit per sample (for BCEWithLogitsLoss)
    The last hidden state of the final LSTM layer is fed into a
    two-layer FC head.
    """
    def __init__(self, n_features: int, hidden_size: int,
                 n_layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, features)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]      # take last timestep
        return self.head(last_hidden).squeeze(-1)

## 3. Hyperparameter Search
Random search over lookback, hidden size, number of LSTM layers, dropout, learning rate, and batch size using TimeSeriesSplit (5 folds). Primary optimisation target: validation F1.

In [ ]:
PARAM_GRID = {
    "lookback":     [24, 48, 72, 168],
    "hidden_size":  [64, 128, 256],
    "n_layers":     [1, 2],
    "dropout":      [0.2, 0.3],
    "lr":           [1e-3, 5e-4, 1e-4],
    "batch_size":   [64, 128],
}
N_TRIALS = 15

train_val = pd.concat([train, val], ignore_index=True)

def build_fn(params, fold_train_df):
    pw        = compute_pos_weight(fold_train_df[TARGET])
    model     = LSTMClassifier(N_FEATURES, params["hidden_size"],
                                params["n_layers"], params["dropout"])
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw.to(DEVICE))
    return model, optimizer, criterion

search_results = random_search(
    PARAM_GRID, N_TRIALS, build_fn, train_val, feature_cols, DEVICE,
    use_sequences=True, n_cv_splits=5, max_epochs=25, patience=5
)

print("\n\u2500\u2500 Top 5 Configurations \u2500\u2500")
for r in search_results[:5]:
    print(f"  F1={r['mean_cv_f1']:.4f}  {r['params']}")

In [ ]:
best_params = search_results[0]["params"]
print("Best hyperparameters:", best_params)

## 4. Final Training

In [ ]:
final_scaler, final_cont_cols = fit_scaler(train, feature_cols)
train_sf = apply_scaler(train, final_scaler, final_cont_cols)
val_sf   = apply_scaler(val,   final_scaler, final_cont_cols)
test_sf  = apply_scaler(test,  final_scaler, final_cont_cols)

LB = best_params["lookback"]
BS = best_params["batch_size"]

train_ds_f = SequenceDataset(train_sf, feature_cols, LB)
val_ds_f   = SequenceDataset(val_sf,   feature_cols, LB)
test_ds_f  = SequenceDataset(test_sf,  feature_cols, LB)

train_loader_f = DataLoader(train_ds_f, batch_size=BS, shuffle=True, drop_last=True)
val_loader_f   = DataLoader(val_ds_f,   batch_size=BS * 2, shuffle=False)
test_loader_f  = DataLoader(test_ds_f,  batch_size=BS * 2, shuffle=False)

final_pw = compute_pos_weight(train[TARGET])
final_model = LSTMClassifier(N_FEATURES, best_params["hidden_size"],
                              best_params["n_layers"], best_params["dropout"]).to(DEVICE)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
final_criterion = nn.BCEWithLogitsLoss(pos_weight=final_pw.to(DEVICE))

history, best_val_f1 = train_model(
    final_model, train_loader_f, val_loader_f,
    final_optimizer, final_criterion, DEVICE,
    max_epochs=50, patience=10, checkpoint_dir=CHECKPOINT_DIR
)
print(f"\nBest validation F1: {best_val_f1:.4f}")

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["loss"],       label="Val loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(hist_df["epoch"], hist_df["f1"],  label="Val F1")
axes[1].plot(hist_df["epoch"], hist_df["auc"], label="Val AUC")
axes[1].set_title("Validation Metrics"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle("LSTM Training History"); plt.tight_layout(); plt.show()

## 5. Evaluation

In [ ]:
best_model = LSTMClassifier(N_FEATURES, best_params["hidden_size"],
                             best_params["n_layers"], best_params["dropout"]).to(DEVICE)
best_model.load_state_dict(torch.load(CHECKPOINT_DIR / "best_model.pt", map_location=DEVICE))

val_metrics  = evaluate(best_model, val_loader_f,  final_criterion, DEVICE)
test_metrics = evaluate(best_model, test_loader_f, final_criterion, DEVICE)

results_df = pd.DataFrame([val_metrics, test_metrics], index=["Validation", "Test"])
print(results_df.round(4))

In [ ]:
def get_preds_probs(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            all_logits.append(model(X.to(device)).cpu())
            all_labels.append(y)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    probs  = 1 / (1 + np.exp(-logits))
    return labels, probs, (probs >= 0.5).astype(int)

val_labels,  val_probs,  val_preds  = get_preds_probs(best_model, val_loader_f,  DEVICE)
test_labels, test_probs, test_preds = get_preds_probs(best_model, test_loader_f, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, ax=axes[0],
    display_labels=["No Spike","Spike"], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix \u2014 Test Set (LSTM)")

for labels, probs, name in [(val_labels, val_probs, "Validation"),
                             (test_labels, test_probs, "Test")]:
    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc(fpr,tpr):.3f})")
axes[1].plot([0,1],[0,1],"k--",lw=0.8)
axes[1].set_title("ROC Curve \u2014 LSTM"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].legend()
plt.tight_layout(); plt.show()

print("\nTest Classification Report:")
print(classification_report(test_labels, test_preds, target_names=["No Spike","Spike"]))

## 6. Summary

In [ ]:
print("=" * 50)
print("LSTM FINAL RESULTS")
print("=" * 50)
print(f"Best hyperparameters : {best_params}")
print(f"\nValidation set:")
for k, v in val_metrics.items(): print(f"  {k:12s}: {v:.4f}")
print(f"\nTest set (out-of-sample):")
for k, v in test_metrics.items(): print(f"  {k:12s}: {v:.4f}")
print(f"\nNote: LSTM used {N_FEATURES} features (manual lags dropped)")
print(f"Checkpoints saved to: {CHECKPOINT_DIR.resolve()}")